# 02 — Sentinel-2 Multi-Dataset Preprocessing Pipeline
## Generalized Dataset Ingestion, Abstract Base Classes & Universal Preprocessor Integration

### Overview
This pipeline generalizes Sentinel-2 cloud removal to multiple heterogeneous datasets using:
- **`BaseDataset(ABC, Dataset)`**: Abstract base class decoupling source folder layouts from downstream processing.
- **Specialized Loaders**:
  - `DatasetALoader`: Flat CSV coordinates (Urban / Agri).
  - `DatasetBLoader`: Nested yearly/regional time series (`forestry_timeseries.json`), flattened into single converted TIFFs.
  - `DatasetCLoader`: FeatureCollection GeoJSON (`highlands.geojson`).
- **Standardized Output Layout**: Strictly **256x256** GeoTIFFs under `output/<dataset_name>/<sample_id>/`:
  - `composite_reflectance.tif` (256x256)
  - `composite_norm.tif` (256x256)
  - Numbered band subfolders `1/`, `2/`, `3/`, `4/` (each 256x256)
  - `comparison.jpg` (3-panel preview for that coordinate)
  - `metadata.json`
- **Hard Cap**: `MAX_SAMPLES_PER_DATASET = 5` strictly enforced across all datasets.


## 1. Environment Setup & Imports


In [1]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import tifffile
import torch
from torch.utils.data import DataLoader, Dataset
from PIL import Image

from multi_dataset_pipeline import (
    BaseDataset,
    DatasetALoader,
    DatasetBLoader,
    DatasetCLoader,
    MultiPipelineConfig,
    run_multi_pipeline,
)

print(f"PyTorch version: {torch.__version__}")
print(f"TiffFile ver   : {tifffile.__version__}")


c:\Users\advik\AppData\Local\Programs\Python\Python313\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


PyTorch version: 2.14.0+cpu
TiffFile ver   : 2026.9.20


## 2. Configuration & Dataset Loaders
Loads heterogeneous datasets from `input/` and targets `output/`.


In [2]:
config = MultiPipelineConfig(
    input_dir=Path("input"),
    output_root=Path("output"),
    max_samples_per_dataset=5
)

loaders = [
    DatasetALoader(config.input_dir / "dataset_a"),
    DatasetBLoader(config.input_dir / "dataset_b"),
    DatasetCLoader(config.input_dir / "dataset_c"),
]

for l in loaders:
    s0 = l[0]
    print(f"Dataset '{l.dataset_name}': {len(l)} targets | Sample 0: {s0['sample_id']} (Coords: {s0['latitude']}, {s0['longitude']})")

2026-09-28 00:19:47,758 [INFO] [dataset_a] Indexed 5 validated rectangular sample(s) from input\dataset_a
2026-09-28 00:19:47,762 [INFO] [dataset_b] Indexed 5 validated rectangular sample(s) from input\dataset_b
2026-09-28 00:19:47,765 [INFO] [dataset_c] Indexed 5 validated rectangular sample(s) from input\dataset_c
Dataset 'dataset_a': 5 targets | Sample 0: A_01 (Coords: 11.0268, 77.0058)
Dataset 'dataset_b': 5 targets | Sample 0: B_01 (Coords: 10.325, 76.955)
Dataset 'dataset_c': 5 targets | Sample 0: C_01 (Coords: 11.411999999999999, 76.711)


## 3. Execute Multi-Dataset Pipeline (Capped at 5 Samples Per Dataset)


In [3]:
all_results = run_multi_pipeline(config)


2026-09-28 00:19:47,786 [INFO] ================================================================================
2026-09-28 00:19:47,788 [INFO] STARTING MULTI-DATASET SENTINEL-2 PREPROCESSING PIPELINE (LIVE SATELLITE MODE)
2026-09-28 00:19:47,793 [INFO] Input Directory          : C:\Users\advik\OneDrive\Desktop\ADVIK FILES\Skills\coding\projects\Ours\SIH\v3\input
2026-09-28 00:19:47,798 [INFO] Output Directory         : C:\Users\advik\OneDrive\Desktop\ADVIK FILES\Skills\coding\projects\Ours\SIH\v3\output
2026-09-28 00:19:47,799 [INFO] Output Size              : (256, 256) (Strict 256x256)
2026-09-28 00:19:47,800 [INFO] Cap Per Dataset          : MAX_SAMPLES_PER_DATASET = 5
2026-09-28 00:19:47,800 [INFO] Speed Settings           : max_dates = 3, threads = 3
2026-09-28 00:19:47,802 [INFO] ================================================================================
2026-09-28 00:19:47,804 [INFO] [dataset_a] Indexed 5 validated rectangular sample(s) from input\dataset_a
2026-09-28 00:19

## 4. Visual Verification Across Datasets (`comparison.jpg`)


In [4]:
for d_name, d_res in all_results.items():
    print(f"=== Visual Preview: {d_name} ===")
    sample_to_show = next((r for r in d_res if r["status"] == "SUCCESS"), None)
    if sample_to_show:
        jpg_p = sample_to_show["result"]["comparison_path"]
        img = Image.open(jpg_p)
        plt.figure(figsize=(15, 5))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Dataset '{d_name}' | {sample_to_show['sample_id']}", fontsize=12, fontweight="bold")
        plt.show()


=== Visual Preview: dataset_a ===
=== Visual Preview: dataset_b ===
=== Visual Preview: dataset_c ===


C:\Users\advik\AppData\Local\Temp\ipykernel_31508\444581481.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Multi-Dataset Validation & 256x256 Dimension Checks


In [5]:
print("=" * 85)
print(f"{'Dataset':<15} {'Sample ID':<15} {'Composite Shape':<18} {'Comparison File':<20} {'Status'}")
print("=" * 85)
for d_name, d_res in all_results.items():
    for r in d_res:
        if r["status"] == "SUCCESS":
            tif_p = Path(r["result"]["tif_paths"][0])
            arr = tifffile.imread(str(tif_p))
            shape_str = str(arr.shape)
            is_256 = arr.shape[-2:] == (256, 256)
            jpg_name = Path(r["result"]["comparison_path"]).name
            print(f"{d_name:<15} {r['sample_id']:<15} {shape_str:<18} {jpg_name:<20} {'256x256 VALID' if is_256 else 'INVALID'}")
print("=" * 85)


Dataset         Sample ID       Composite Shape    Comparison File      Status
dataset_a       A_01            (4, 256, 256)      comparison.jpg       256x256 VALID
dataset_a       A_02            (4, 256, 256)      comparison.jpg       256x256 VALID
dataset_a       A_03            (4, 256, 256)      comparison.jpg       256x256 VALID
dataset_a       A_04            (4, 256, 256)      comparison.jpg       256x256 VALID
dataset_a       A_05            (4, 256, 256)      comparison.jpg       256x256 VALID
dataset_b       B_01            (4, 256, 256)      comparison.jpg       256x256 VALID
dataset_b       B_02            (4, 256, 256)      comparison.jpg       256x256 VALID
dataset_b       B_03            (4, 256, 256)      comparison.jpg       256x256 VALID
dataset_b       B_04            (4, 256, 256)      comparison.jpg       256x256 VALID
dataset_b       B_05            (4, 256, 256)      comparison.jpg       256x256 VALID
dataset_c       C_01            (4, 256, 256)      comparison